In [2]:

import json
from collections import Counter

# ============================================================
# DATASET PATH
# ============================================================

DATASET_PATH = "./dataset/finalV1A/V1A.jsonl"

# ============================================================
# COUNTERS
# ============================================================

length_counter = Counter()
speaker_counter = Counter()

total = 0
error_count = 0
empty_conversation_count = 0


# ============================================================
# COUNT CONVERSATION TURNS
# ============================================================

def count_messages(conversation):
    count = 0

    for line in conversation.splitlines():
        line = line.strip()

        if line and ":" in line:
            count += 1

    return count


# ============================================================
# GET LAST SPEAKER
# ============================================================

def get_last_speaker(conversation):
    lines = [
        line.strip()
        for line in conversation.splitlines()
        if line.strip()
    ]

    if not lines:
        return "Other"

    last_line = lines[-1]

    if last_line.startswith("User:"):
        return "User"

    # Other includes Unknown and every non-User speaker.
    return "Other"


# ============================================================
# READ DATASET
# ============================================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:

    for line_num, line in enumerate(f, start=1):

        line = line.strip()

        if not line:
            continue

        try:
            sample = json.loads(line)

            # Handle double-encoded JSON
            if isinstance(sample, str):
                sample = json.loads(sample)

            conversation = sample["input"]["conversation"]

            if not isinstance(conversation, str):
                raise ValueError("Conversation is not a string")

            total += 1

            # ------------------------------------------------
            # 1. CONVERSATION TURN COUNT
            # ------------------------------------------------

            messages = count_messages(conversation)

            if messages == 0:
                empty_conversation_count += 1
                length_counter["Other"] += 1

            elif messages <= 2:
                length_counter["Extra Small (1-2)"] += 1

            elif messages <= 6:
                length_counter["Small (3-6)"] += 1

            elif messages <= 15:
                length_counter["Medium (7-15)"] += 1

            elif messages <= 30:
                length_counter["Large (16-30)"] += 1

            else:
                length_counter["Extra Large (31+)"] += 1

            # ------------------------------------------------
            # 2. LAST SPEAKER COUNT
            # ------------------------------------------------

            last_speaker = get_last_speaker(conversation)
            speaker_counter[last_speaker] += 1

        except Exception as e:
            error_count += 1
            print(f"Error at line {line_num}: {e}")


# ============================================================
# OUTPUT 1: CONVERSATION TURN COUNT
# ============================================================

print("\n" + "=" * 65)
print("1. CONVERSATION TURN COUNT")
print("=" * 65)

print(f"Total Valid Records: {total}")
print(f"Invalid Records    : {error_count}")
print("-" * 65)

categories = [
    "Extra Small (1-2)",
    "Small (3-6)",
    "Medium (7-15)",
    "Large (16-30)",
    "Extra Large (31+)",
    "Other"
]

for category in categories:
    count = length_counter[category]
    percentage = count / total * 100 if total else 0

    print(f"{category:25}: {count:8} ({percentage:6.2f}%)")


# ============================================================
# OUTPUT 2: LAST CONVERSATION SPEAKER
# ============================================================

print("\n" + "=" * 65)
print("2. LAST CONVERSATION SPEAKER COUNT")
print("=" * 65)

print(f"{'Last Speaker':25} {'Count':>10} {'Percentage':>15}")
print("-" * 65)

for speaker in ["User", "Other"]:
    count = speaker_counter[speaker]
    percentage = count / total * 100 if total else 0

    print(
        f"{speaker:25} "
        f"{count:10} "
        f"{percentage:14.2f}%"
    )

print("-" * 65)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)

print(f"Total Valid Records       : {total}")
print(f"Invalid Records           : {error_count}")
print(f"Empty Conversations       : {empty_conversation_count}")
print(f"Last message = User       : {speaker_counter['User']}")
print(f"Last message = Other      : {speaker_counter['Other']}")
print("=" * 65)



1. CONVERSATION TURN COUNT
Total Valid Records: 26456
Invalid Records    : 0
-----------------------------------------------------------------
Extra Small (1-2)        :       14 (  0.05%)
Small (3-6)              :     4765 ( 18.01%)
Medium (7-15)            :     7940 ( 30.01%)
Large (16-30)            :    13737 ( 51.92%)
Extra Large (31+)        :        0 (  0.00%)
Other                    :        0 (  0.00%)

2. LAST CONVERSATION SPEAKER COUNT
Last Speaker                   Count      Percentage
-----------------------------------------------------------------
User                           18750          70.87%
Other                           7706          29.13%
-----------------------------------------------------------------

FINAL SUMMARY
Total Valid Records       : 26456
Invalid Records           : 0
Empty Conversations       : 0
Last message = User       : 18750
Last message = Other      : 7706
